# ForgeAudio (`audio-forge`) — Remote A100/L4 GPU Worker

This notebook runs stem separation on a **Google Colab A100 / L4 GPU** and exposes a FastAPI endpoint that the local Rust `forge` CLI and `forge-mcp` server call when `FORGE_COLAB_WORKER_URL` is set.

**Implemented:** `POST /api/v1/execute` with `task: separate_stems` runs a **BS-RoFormer** vocal/instrumental model (`model_bs_roformer_ep_317_sdr_12.9755.ckpt` by default, or `payload.model`) through `audio-separator` and returns two stems, `vocals` and `other` (the instrumental). Drums, bass, guitar and keys are not separated here; use the Rust DSP splitter (`FORGE_WORKER_BACKEND=dsp`) when you need them.

**Not implemented:** 6-stem separation, Apollo restoration, YuE2-3B and ACE-Step generation, SheetSage2 / Basic-Pitch transcription, and Seed-VC / RVC conversion. Any other `task` is rejected with HTTP 400.

In [ ]:
!pip install -q fastapi uvicorn[standard] audio-separator[gpu] soundfile librosa numpy pyngrok huggingface_hub torch torchaudio

In [ ]:
import base64, hmac, os, json, subprocess, tempfile
from pathlib import Path
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel
import torch

app = FastAPI(title="ForgeAudio Colab A100 Worker")
ALLOWED_MODELS = frozenset({
    "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    "model_mel_band_roformer_kim_vocals.ckpt",
})
MAX_UPLOAD_BYTES = 256 * 1024 * 1024
API_TOKEN = os.environ.get("FORGE_COLAB_API_TOKEN", "").strip()
if not API_TOKEN:
    raise RuntimeError("FORGE_COLAB_API_TOKEN must be set before starting the worker")

def require_bearer(authorization: str | None = Header(default=None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing Bearer token")
    token = authorization[7:].strip()
    if not hmac.compare_digest(token, API_TOKEN):
        raise HTTPException(status_code=403, detail="Invalid Bearer token")

class WorkerRequest(BaseModel):
    task: str
    payload: dict

@app.get("/health")
def health():
    return {
        "ok": True,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 0
    }

def _stem_tag(filename):
    # audio-separator names outputs "<input>_(<Stem>)_<model>.wav"; only the last group counts.
    name = Path(filename).name
    start = name.rfind("_(")
    end = name.find(")", start + 2) if start >= 0 else -1
    return "".join(c for c in name[start + 2:end].lower() if c.isalnum()) if end >= 0 else None

STEM_ROLES = {"vocals": {"vocals", "vocal"}, "other": {"instrumental", "instrumentals"}}

@app.post("/api/v1/execute", dependencies=[Depends(require_bearer)])
def execute(req: WorkerRequest):
    if req.task != "separate_stems":
        raise HTTPException(status_code=400, detail=f"Unknown task: {req.task}")
    model = req.payload.get("model", "model_bs_roformer_ep_317_sdr_12.9755.ckpt")
    if not isinstance(model, str) or model not in ALLOWED_MODELS:
        raise HTTPException(status_code=400, detail=f"Unsupported model checkpoint: {model!r}")
    audio_b64 = req.payload.get("audio_b64")
    if not isinstance(audio_b64, str) or len(audio_b64) > (MAX_UPLOAD_BYTES * 4) // 3 + 4:
        raise HTTPException(status_code=413, detail="Encoded audio payload missing or too large")
    try:
        raw_audio = base64.b64decode(audio_b64, validate=True)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid base64 audio payload: {exc}") from exc
    if len(raw_audio) > MAX_UPLOAD_BYTES:
        raise HTTPException(status_code=413, detail="Decoded audio exceeds 256 MiB limit")
    from audio_separator.separator import Separator
    with tempfile.TemporaryDirectory(prefix="forge_") as tmp:
        work = Path(tmp).resolve()
        ext = "".join(c for c in Path(str(req.payload.get("filename", "input.wav"))).suffix.lower() if c.isalnum())
        src = work / f"input.{ext or 'wav'}"
        src.write_bytes(raw_audio)
        sep = Separator(output_dir=str(work), output_format="WAV")
        sep.load_model(model_filename=model)
        stems = {}
        for f in sep.separate(str(src)):
            cand = Path(f) if Path(f).is_absolute() else work / f
            resolved = cand.resolve()
            if not resolved.is_relative_to(work) or not resolved.is_file():
                raise HTTPException(status_code=500, detail="Separator wrote stem outside temp directory")
            for role, tags in STEM_ROLES.items():
                if role not in stems and _stem_tag(f) in tags:
                    stems[role] = base64.b64encode(resolved.read_bytes()).decode("ascii")
        if not stems:
            raise HTTPException(status_code=500, detail="separator produced no vocal or instrumental stem")
    return {"ok": True, "task": req.task, "data": {"stems": stems}}